# R/C Fixed-Wing Design Notebook
The full sizing chain from **learnrcfixedwing** as runnable Python. Change the three inputs, run all cells, watch the airplane change.

Companion to [the guide](https://akanwar.github.io/learnrcfixedwing/) — chapters 2–4 explain every line.

> **Accuracy note:** drafted with AI assistance and reviewed by a human, but it will contain mistakes. Verify against the guide's linked references before you build. Corrections: open a PR at [github.com/akanwar/learnrcfixedwing](https://github.com/akanwar/learnrcfixedwing).

In [ ]:
# ----- THE THREE HONEST INPUTS -----
mass_g   = 1500    # all-up weight, grams (with battery!)
cruise_v = 12.0    # cruise speed, m/s
span_mm  = 1470    # wingspan, mm

# ----- advanced knobs (trainer defaults) -----
AR      = 6.5      # aspect ratio, span^2 / area
CLmax   = 1.1      # wing max lift coefficient
arm_f   = 0.45     # tail arm as a fraction of span
Vh, Vv  = 0.55, 0.04   # tail volume coefficients
Wkg     = 150      # power density, W per kg
e       = 0.8      # span efficiency (induced drag)
RHO, G  = 1.225, 9.81

In [ ]:
import math
m = mass_g/1000; W = m*G; b = span_mm/1000
S  = b*b/AR              # wing area, m^2
c  = S/b                 # mean chord, m
wl = mass_g/(S*100)      # wing loading, g/dm^2
Vs   = math.sqrt(2*W/(RHO*S*CLmax))
CL   = 2*W/(RHO*cruise_v**2*S)
Cdi  = CL**2/(math.pi*AR*e)
l    = arm_f*b
Sh, Sv = Vh*S*c/l, Vv*S*b/l
bh = math.sqrt(4*Sh); ch = Sh/bh          # h-tail at AR 4
hv = math.sqrt(1.5*Sv); cv = Sv/hv        # fin at AR 1.5
P  = Wkg*m
Re_cruise, Re_land = 68000*cruise_v*c, 68000*1.3*Vs*c
print(f'Wing:   S = {S*100:5.1f} dm2   loading = {wl:4.1f} g/dm2   chord = {c*1000:4.0f} mm')
print(f'Speeds: stall = {Vs:4.1f} m/s   approach = {1.3*Vs:4.1f} m/s   cruise CL = {CL:4.2f}   Cd,i = {Cdi:.4f}')
print(f'Tail:   arm = {l*1000:4.0f} mm   Sh = {Sh*1e4:4.0f} cm2 ({bh*1000:.0f} x {ch*1000:.0f})   Sv = {Sv*1e4:4.0f} cm2 ({hv*1000:.0f} tall x {cv*1000:.0f})')
print(f'Ctrl:   aileron {0.2*c*1000:.0f} x {0.45*b/2*1000:.0f} mm   elevator {0.3*ch*1000:.0f} mm   rudder {0.35*cv*1000:.0f} mm')
print(f'Power:  {P:.0f} W   ~{P/11.1:.1f} A on 3S   CG (maiden) = {0.25*c*1000:.0f} mm aft of LE')
print(f'Re:     cruise {Re_cruise/1000:.0f}k   landing {Re_land/1000:.0f}k' + ('   << low-Re trouble zone!' if Re_land < 1e5 else ''))

## Your turn
1. Add 200 g (a bigger battery). Which numbers moved, and by how much?
2. Halve the aspect ratio at the same span — wait, you can't: `S` would double. Hold `S` instead: what happens to induced drag?
3. Plot stall speed against wing loading (`matplotlib`) and mark your design on the curve, like the guide's calculator does.

In [ ]:
# a starter for exercise 3
import numpy as np, matplotlib.pyplot as plt
wls = np.linspace(15, 90, 60)                       # g/dm^2
vss = np.sqrt(2*(wls*0.1*G)/(RHO*CLmax))
plt.plot(wls, vss); plt.scatter([wl],[Vs], color='red', zorder=5)
plt.axvspan(25, 60, alpha=.12, color='green', label='trainer territory')
plt.xlabel('wing loading, g/dm$^2$'); plt.ylabel('stall speed, m/s'); plt.legend(); plt.show()